# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field and field IDs.

> Each entity is referenced by its `@id` as per the dataset schema.

In [ ]:
# List all record sets (referenced by @id)
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"   - {fld.name} (@id: {fld.id}, type: {fld.data_type})")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

In [ ]:
# If the dataset contains multiple record sets, list them for selection
record_set_ids = [rs.id for rs in record_sets]
print(f"Available record set @ids: {record_set_ids}")

# For this dataset, we expect the main data to be in the primary record set
# Let's select the first one for demonstration
main_record_set_id = record_set_ids[0]

# Load records from the primary record set by @id
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print(f"First 5 columns: {df.columns[:5].tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping on numeric or categorical fields.

Below, we'll select a numeric field by its `@id` for demonstration. Adjust the field ids to those from your dataset.

In [ ]:
# Example: select a numeric field for filtering and normalization
# First, list numeric fields by their @id for clarity

numeric_fields = [fld for fld in record_sets[0].fields if fld.data_type in ['Number', 'Float', 'Integer']]
print("Numeric fields available:")
for nf in numeric_fields:
    print(f"  {nf.name} (@id: {nf.id})")

# For demonstration, use the first numeric field found.
numeric_field_id = numeric_fields[0].id if numeric_fields else None
print(f"\nUsing numeric field '@id': {numeric_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (showing up to 5 records):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f'{numeric_field_id}_normalized'] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Try grouping by a categorical field if available
    categorical_fields = [fld for fld in record_sets[0].fields if fld.data_type in ['Text', 'String']]
    group_field_id = categorical_fields[0].id if categorical_fields else None
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print('No numeric field found or field not present in DataFrame.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the chosen numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If categorical group field is present, plot the average
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset schema and data using `mlcroissant`, referencing entities by their `@id` for clarity and reproducibility.
- The dataset contains multiple fields suitable for numeric and categorical processing.
- Filtering, normalization, and basic grouping were performed as initial exploratory analyses.
- Basic distribution visualizations for the primary numeric field were plotted.

> For further analysis, tailor filtering and grouping fields to the biological or experimental questions at hand.